# MilaanAI — Fine-tune Qwen2.5-3B as a Transaction Categorizer

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run cells top to bottom. Total time on a free T4: ~45-75 minutes.

Pipeline: upload data → load 4-bit Qwen2.5-3B → LoRA → train → predict on
test set → download `student_preds.jsonl` (grade it locally with
`python cli.py eval-cat --preds student_preds.jsonl`).

In [ ]:
%%capture
# 1) Install Unsloth (takes ~2 min)
!pip install unsloth

In [ ]:
# 2) Upload your three dataset files when prompted:
#    data/finetune/train.jsonl, val.jsonl, test.jsonl  (from your laptop)
from google.colab import files
up = files.upload()
assert {"train.jsonl", "val.jsonl", "test.jsonl"} <= set(up.keys()), \
    "Please upload all three: train.jsonl, val.jsonl, test.jsonl"
print("uploaded:", list(up.keys()))

In [ ]:
# 3) Load Qwen2.5-3B-Instruct in 4-bit (QLoRA)
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 512   # our inputs are short narration strings

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ,
    dtype=None,          # auto
    load_in_4bit=True,
)
print("loaded:", model.config._name_or_path)

In [ ]:
# 4) Attach LoRA adapters (~1-2% of params trained)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 5) Build the training dataset with Qwen's chat template
import json
from datasets import Dataset

CATEGORIES = ['sales_receipt', 'vendor_payment', 'salary', 'rent', 'bank_charges', 'gst_tds_payment', 'loan_emi', 'utility_bill', 'upi_p2p', 'refund', 'interest_credit', 'cash_deposit', 'cash_withdrawal', 'insurance', 'other']
CATS_STR = ", ".join(CATEGORIES)

SYS = ("You classify Indian bank statement transactions into exactly one "
       "category. Reply with only the category name.")


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]


def to_chat(row, with_answer=True):
    msgs = [
        {"role": "system", "content": SYS},
        {"role": "user", "content":
          f"Categories: {CATS_STR}\nTransaction: {row['text']}\nCategory:"},
    ]
    if with_answer:
        msgs.append({"role": "assistant", "content": row["category"]})
    return msgs


train_rows = load_jsonl("train.jsonl")
train_texts = [tokenizer.apply_chat_template(to_chat(r), tokenize=False)
               for r in train_rows]
train_ds = Dataset.from_dict({"text": train_texts})
print(len(train_ds), "training examples")
print(train_texts[0][:400])

In [ ]:
# 6) Train (~30-50 min on T4). Loss should fall well below 0.1.
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=20,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        warmup_steps=20,
        seed=42,
        output_dir="out",
        report_to="none",
        max_seq_length=MAX_SEQ,
    ),
)
stats = trainer.train()
print(stats)

In [ ]:
# 7) Predict on the TEST set -> student_preds.jsonl
import re
from tqdm import tqdm

FastLanguageModel.for_inference(model)
test_rows = load_jsonl("test.jsonl")

def predict_batch(rows):
    prompts = [tokenizer.apply_chat_template(
        to_chat(r, with_answer=False), tokenize=False,
        add_generation_prompt=True) for r in rows]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                       truncation=True, max_length=MAX_SEQ).to("cuda")
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    texts = tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True)
    preds = []
    for t in texts:
        t = t.strip().lower()
        match = next((c for c in CATEGORIES if c in t), "other")
        preds.append(match)
    return preds

tokenizer.padding_side = "left"
BATCH = 32
with open("student_preds.jsonl", "w", encoding="utf-8") as f:
    correct = 0
    for i in tqdm(range(0, len(test_rows), BATCH)):
        batch = test_rows[i:i+BATCH]
        for row, pred in zip(batch, predict_batch(batch)):
            f.write(json.dumps({"text": row["text"], "pred": pred}) + "\n")
            correct += (pred == row["category"])
print(f"STUDENT TEST ACCURACY: {correct/len(test_rows):.4f}  "
      f"({correct}/{len(test_rows)})")

In [ ]:
# 8) Download predictions (grade locally: python cli.py eval-cat --preds student_preds.jsonl)
from google.colab import files
files.download("student_preds.jsonl")

In [ ]:
# 9) Save LoRA adapters (small, ~60MB) so you never retrain from scratch
model.save_pretrained("milaan_categorizer_lora")
tokenizer.save_pretrained("milaan_categorizer_lora")
!zip -r milaan_categorizer_lora.zip milaan_categorizer_lora
files.download("milaan_categorizer_lora.zip")

## Optional — export GGUF for local laptop inference (slow: ~20-30 min)

Only run this if you want the "runs on my own laptop via Ollama" demo.
It compiles llama.cpp inside Colab and produces a ~2GB q4_k_m file.

In [ ]:
# 10) OPTIONAL: GGUF export for Ollama / llama.cpp on your laptop
model.save_pretrained_gguf("milaan_gguf", tokenizer,
                           quantization_method="q4_k_m")
import glob
print(glob.glob("milaan_gguf/*"))
# Download the .gguf file from the Files panel on the left (too big for files.download sometimes)